In [3]:
from autogen_core.models import UserMessage
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
load_dotenv()
import os
import asyncio

api_key = os.getenv('OPENAI_API_KEY')
openai_model_client = OpenAIChatCompletionClient(
    model = "gpt-4o",
    api_key=api_key
)

In [7]:
assistant = AssistantAgent(name = "assistant",model_client = openai_model_client)

In [8]:
result = await assistant.run(task = "What is the capital of france")
print(result)

messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 16, 18, 4, 20, 152817, tzinfo=datetime.timezone.utc), content='What is the capital of france', type='TextMessage'), TextMessage(source='assistant', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=9), metadata={}, created_at=datetime.datetime(2025, 6, 16, 18, 4, 21, 742169, tzinfo=datetime.timezone.utc), content='The capital of France is Paris. TERMINATE', type='TextMessage')] stop_reason=None


In [11]:
assistant2 = AssistantAgent(name = 'Assistant',
                            model_client=openai_model_client,
                            description = 'Give output in JSON',
                            system_message='You are a helpful assistant that provide accurate information about history')

In [12]:
result = await assistant2.run(task = "What is the capital of france")
print(result)

messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 16, 18, 7, 41, 106088, tzinfo=datetime.timezone.utc), content='What is the capital of france', type='TextMessage'), TextMessage(source='Assistant', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=7), metadata={}, created_at=datetime.datetime(2025, 6, 16, 18, 7, 43, 298198, tzinfo=datetime.timezone.utc), content='The capital of France is Paris.', type='TextMessage')] stop_reason=None


In [14]:
result.messages[-1].content

'The capital of France is Paris.'

In [20]:
from io import BytesIO

import requests
from autogen_agentchat.messages import MultiModalMessage
from autogen_core import Image as AGImage
from PIL import Image

pil_image = Image.open(BytesIO(requests.get("https://picsum.photos/300/200").content))
img = AGImage(pil_image)
multi_modal_message = MultiModalMessage(content=["Can you describe the content of this image?", img], source="User")
img.image.show()
result = await assistant2.run(task = multi_modal_message)
result.messages[-1].content

'The image shows a scenic view of mountains reflected in a calm body of water. The sky is partly cloudy with patches of blue, creating a serene and picturesque landscape. The mountains are partially covered with snow or clouds, adding to the dramatic effect.'

In [ ]:
multi_modal_message

MultiModalMessage(source='User', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 16, 19, 7, 43, 370373, tzinfo=datetime.timezone.utc), content=['Can you describe the content of this image?', <autogen_core._image.Image object at 0x000001DCE1C14650>], type='MultiModalMessage')

In [17]:
result = await assistant2.run(task = multi_modal_message)

In [18]:
result.messages[-1].content

'The image shows the Statue of Liberty against a background of clouds and blue sky. The statue is a symbol of freedom and is located on Liberty Island in New York Harbor, USA.'

In [7]:
def get_weather(city:str)->str:
    return f"The weather in {city} is sunny with temperature of 25 degree celcius."

weather_agent = AssistantAgent(
    name = "weather_agent",
    model_client=openai_model_client,
    description = "you are a weather agent that provides accurate weather information.",
    system_message="You are  a helpful assitant that provides accurate weather informationby using get_weather",
    tools=[get_weather] 
)
result = await weather_agent.run(task = "what is the waether in Delhi")
result.messages[-1].content

'The weather in Delhi is sunny with temperature of 25 degree celcius.'

In [13]:
from pydantic import BaseModel

class Information(BaseModel):
    fact:str
    source:str

model_client_structured = OpenAIChatCompletionClient(model="gpt-4o",api_key=api_key,response_format=Information)

structured_output_agent = AssistantAgent(
    name = "assistant",
    model_client= model_client_structured,
    #output_content_type=Information,
    system_message="You are a helpful assistant that provide accurate information about history.",

)
result = await structured_output_agent.run(task = "What is the capital of india?")
print(result.messages[-1].content)



{"fact":"New Delhi is the capital of India.","source":"According to the Government of India's official portal and various geographical references, New Delhi is the capital city of India."}
